# Nationality Prediction from First and Last Names

This notebook uses the `name2nat` model (included in this repository) to predict the nationality origin of names from `data/firstname.csv` and `data/lastname.csv`.

Each output CSV retains the original columns and adds:
- `predicted_nationality` — the top-1 predicted nationality
- `confidence` — the model's confidence score (0–1)

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import pandas as pd
from name2nat import Name2nat
from tqdm.auto import tqdm

c:\Users\Cheng\anaconda3\envs\name2nat\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.1 is required but found 1.13.1
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
model = Name2nat()
print('Model loaded successfully.')

2026-06-21 20:12:16,471 loading file c:\Users\Cheng\Documents\GitHub\name2nat\name2nat/best-model.pt
Model loaded successfully.


In [3]:
def predict_nationalities(names, model, batch_size=1024):
    """Predict nationality for a list of names, processing in batches."""
    nationalities = [None] * len(names)
    confidences = [None] * len(names)

    valid_indices = [i for i, n in enumerate(names) if isinstance(n, str) and n.strip()]
    valid_names = [names[i] for i in valid_indices]
    print(f'  Valid names: {len(valid_names):,} / {len(names):,}')

    for i in tqdm(range(0, len(valid_names), batch_size), desc='Predicting'):
        batch = valid_names[i:i + batch_size]
        batch_indices = valid_indices[i:i + batch_size]
        results = model(batch, top_n=1, use_dict=True)
        for j, (_, preds) in enumerate(results):
            idx = batch_indices[j]
            if preds:
                nationalities[idx] = preds[0][0]
                confidences[idx] = round(preds[0][1], 6)
    return nationalities, confidences

## First Names

In [4]:
df_first = pd.read_csv('data/firstname.csv')
print(f'Loaded {len(df_first):,} first names')
df_first.head()

Loaded 100,000 first names


,first_name,n_incidents
0,david,1266734
1,michael,1225179
2,daniel,743583
3,john,742067
4,thomas,694262


In [5]:
first_names = df_first['first_name'].tolist()
nat, conf = predict_nationalities(first_names, model)
df_first['predicted_nationality'] = nat
df_first['confidence'] = conf
df_first.head(10)

  Valid names: 99,999 / 100,000


Predicting: 100%|██████████| 98/98 [02:43<00:00,  1.67s/it]


,first_name,n_incidents,predicted_nationality,confidence
0,david,1266734,American,0.249013
1,michael,1225179,American,0.331465
2,daniel,743583,American,0.239577
3,john,742067,American,0.342939
4,thomas,694262,Greek,0.240514
5,peter,673685,American,0.402706
6,robert,650396,American,0.388435
7,andrew,534189,American,0.249378
8,james,520583,American,0.238433
9,paul,491029,American,0.226130


In [6]:
df_first.to_csv('data/firstname_predicted.csv', index=False)
print(f'Saved data/firstname_predicted.csv ({len(df_first):,} rows)')

Saved data/firstname_predicted.csv (100,000 rows)


## Last Names

In [7]:
df_last = pd.read_csv('data/lastname.csv')
print(f'Loaded {len(df_last):,} last names')
df_last.head()

Loaded 100,000 last names


,last_name,n_incidents
0,kim,1326018
1,lee,1229609
2,wang,773720
3,chen,714150
4,li,558551


In [8]:
last_names = df_last['last_name'].tolist()
nat, conf = predict_nationalities(last_names, model)
df_last['predicted_nationality'] = nat
df_last['confidence'] = conf
df_last.head(10)

  Valid names: 99,998 / 100,000


Predicting: 100%|██████████| 98/98 [02:45<00:00,  1.69s/it]


,last_name,n_incidents,predicted_nationality,confidence
0,kim,1326018,Korean,0.268181
1,lee,1229609,American,0.405221
2,wang,773720,Korean,0.569408
3,chen,714150,American,0.214434
4,li,558551,American,0.327726
5,zhang,527528,Chinese,0.805861
6,park,512722,American,0.325639
7,liu,487871,American,0.186728
8,kumar,483084,Indian,0.336817
9,singh,445113,American,0.197021


In [9]:
df_last.to_csv('data/lastname_predicted.csv', index=False)
print(f'Saved data/lastname_predicted.csv ({len(df_last):,} rows)')

Saved data/lastname_predicted.csv (100,000 rows)


## Summary Statistics

In [10]:
for label, df in [('First names', df_first), ('Last names', df_last)]:
    valid = df['confidence'].notna()
    print(f'\n=== {label} ===')
    print(f'Total names: {len(df):,}')
    print(f'Predicted: {valid.sum():,}')
    print(f'Mean confidence: {df["confidence"].mean():.4f}')
    print(f'Median confidence: {df["confidence"].median():.4f}')
    print(f'Names with confidence >= 0.9: {(df["confidence"] >= 0.9).sum():,} ({(df.loc[valid, "confidence"] >= 0.9).mean():.1%})')
    print(f'Names with confidence < 0.5: {(df["confidence"] < 0.5).sum():,} ({(df.loc[valid, "confidence"] < 0.5).mean():.1%})')
    print(f'\nTop 10 predicted nationalities:')
    print(df['predicted_nationality'].value_counts().head(10).to_string())


=== First names ===
Total names: 100,000
Predicted: 99,999
Mean confidence: 0.3403
Median confidence: 0.2665
Names with confidence >= 0.9: 2,735 (2.7%)
Names with confidence < 0.5: 80,493 (80.5%)

Top 10 predicted nationalities:
predicted_nationality
American     31346
Indian       18774
Korean        7021
Japanese      6845
Chinese       6325
Iranian       3419
Brazilian     3295
Thai          2942
Russian       2924
Greek         2464

=== Last names ===
Total names: 100,000
Predicted: 99,998
Mean confidence: 0.3280
Median confidence: 0.2803
Names with confidence >= 0.9: 977 (1.0%)
Names with confidence < 0.5: 85,278 (85.3%)

Top 10 predicted nationalities:
predicted_nationality
American     44316
Indian        8509
Italian       6429
Russian       5113
Japanese      4599
French        4038
Iranian       3915
Greek         3689
German        3253
Brazilian     2851
